In [ ]:
import json
from src.llm import LLM
from src.prompt import *
from src.utils import iterate_schelling_data, coordination_index

prompt_schema = Level0  # prompting technique

In [ ]:
# Get the data
with open('./data/schelling.jsonl', 'r') as file:
    data = json.load(file)

# Get the combinations of problems
problems = iterate_schelling_data(data)

In [ ]:
model = LLM(
    model_id="meta-llama/Llama-3.2-1B-Instruct",
    num_return_sequences=30
)

# Build a flat list of all (idx, problem) prompts
batch_prompts = []
keys = []

for idx in problems:
    for problem in problems[idx]:
        full_prompt = Level0.prefix + problem + Level0.suffix
        batch_prompts.append(full_prompt)
        keys.append((idx, problem))

# Send all prompts in one generate_batch call.
batch_outputs = model.generate_batch(batch_prompts)

# Reconstruct the original nested-dict format
responses = {}
for (idx, problem), prompt_outputs in zip(keys, batch_outputs):
    if idx not in responses:
        responses[idx] = {}
    responses[idx][problem] = prompt_outputs

In [ ]:
logs = []
for idx in responses:
    for problem in responses[idx]:
        log = {
            'idx': idx,
            'prompt': problem,
            "responses": responses[idx][problem]
        }

        logs.append(log)

with open('./logs/schelling_responses.jsonl', 'w') as f:
    json.dump(logs, f, indent=2)

In [ ]:
import re

# Captures everything between <answer>...</answer>
pattern = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.DOTALL)

for idx in responses:
    # Iterate over a fixed list of keys so we can delete from responses[idx] safely
    for response in list(responses[idx].keys()):
        original_list = responses[idx][response]
        filtered_list = []

        # Only keep those elements that contain <answer>...</answer>
        for r in original_list:
            match = pattern.search(r)
            if match:
                filtered_list.append(match.group(1).strip())

        if filtered_list:
            responses[idx][response] = filtered_list
        else:
            del responses[idx][response]

In [ ]:
# Compute the coordination index
results = []
for idx in responses:
    for response in responses[idx]:
        if responses[idx][response]:
            c_index, nc_index = coordination_index(responses[idx][response])

            result = {
                "idx": idx,
                "problem": response,
                "coordination_index": c_index,
                "normalised_coordination_index": nc_index
            }

            results.append(result)

# Write the results to a file
with open("./results/schelling_results.jsonl", "a") as f:
    json.dump(results, f, indent=2)
